# ⚠️ EEG_07b — DEPRECATO

> **Non eseguire questo notebook.**
> I file `dataset_*.pt` che produce hanno `x.shape = (59, 384)` perché
> rimuovono A1/A2 durante il caricamento.
> Il formato corretto per tutta la pipeline è `(61, 384)` — usare **EEG_07e**.

| File prodotto (OBSOLETO) | Motivo eliminazione |
|---|---|
| `dataset_pcc_k6.pt` | 59 canali — pipeline usa 61 |
| `dataset_plv_k6.pt` | 59 canali |
| `dataset_wpli_k6.pt` | 59 canali |
| `dataset_cpcc_abs_k6.pt` | 59 canali |
| `dataset_cpcc_im_k6.pt` | 59 canali |
| `dataset_learned_k6.pt` | 59 canali |
| `dataset_dynamic_k6.pt` | 59 canali |

Sostituiti da `graph_*.pt` e `hgraph_*.pt` costruiti in **EEG_07e** (61 canali, formato
`Data(x, edge_index, y, label_id, subj, sess)`).

Per eliminare i file obsoleti sul server:
```bash
rm data/interim/graphs/dataset_*.pt
```

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from scipy.signal import butter, filtfilt
from scipy.signal import hilbert as sp_hilbert
from torch_geometric.data import Data
import multiprocessing as mp

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme

CSV_ROOT   = project_root / "data" / "raw_csv" / "training_set"
CONFIGS    = project_root / "configs" / "label_schemes"

def parse_folder(folder_name: str):
    """'P003_S002' -> (3, 2)"""
    parts = folder_name.split("_")
    return int(parts[0][1:]), int(parts[1][1:])

ROOT     = project_root
DATA_INT = ROOT / 'data' / 'interim'
FIGURES  = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)

ELOC_PATH  = project_root / "src" / "io" / "ebneuro.locs"
GRAPHS_DIR = project_root / "data" / "interim" / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

K_GRAPH  = 6
SFREQ    = 256
N_CHANS  = 59

# Metodi disponibili:
#   "pcc"      -- Pearson |r| (veloce)
#   "plv"      -- Phase Locking Value (bande theta+alpha)
#   "wpli"     -- Weighted Phase Lag Index (bande theta+alpha)
#   "cpcc_abs" -- CPCCabs = |mean(exp(iDeltaPhi))| (Iacomi et al. 2026)
#   "cpcc_im"  -- CPCCim  = |mean(sin(DeltaPhi))|  (Iacomi et al. 2026)
#   "learned" / "dynamic" -- sentinel edge_index (grafo appreso in forward)
METHODS = ["pcc", "plv", "wpli", "cpcc_abs", "cpcc_im", "learned", "dynamic"]

# Bande freq. per PLV e wPLI (cpcc_abs/cpcc_im usano hilbert broadband)
PLV_BANDS = [(4, 8), (8, 13)]   # theta + alpha

# Forza ricostruzione anche se il .pt esiste gia'
FORCE_REBUILD = False

print(f"project_root : {project_root}")
print(f"Output dir   : {GRAPHS_DIR}")
print(f"K_GRAPH      : {K_GRAPH}  |  SFREQ={SFREQ}")
print(f"Metodi       : {METHODS}")


project_root : /home/daniele_u/miralis-hypergraph-imagined-speech
Output dir   : /home/daniele_u/miralis-hypergraph-imagined-speech/data/interim/graphs
K_GRAPH      : 6  |  SFREQ=256
Metodi       : ['pcc', 'plv', 'wpli', 'cpcc_abs', 'cpcc_im', 'learned', 'dynamic']


In [2]:
import json

def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE     = {"A1", "A2"}
keep_idx    = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
assert len(keep_idx) == N_CHANS

with open(CONFIGS / "label2idx.json") as f:
    word2labelid = json.load(f)

records = []
for sess_dir in sorted(CSV_ROOT.iterdir()):
    if not sess_dir.is_dir(): continue
    subj_id, _ = parse_folder(sess_dir.name)
    trials = sorted([p for p in sess_dir.iterdir() if p.suffix == ".csv"])
    for csv_path in trials:
        word = csv_path.stem.replace("_img", "")
        if word not in word2labelid:
            continue
        label_id = word2labelid[word]
        records.append({
            "path_csv": str(csv_path),
            "label_idx": label_id,
            "subject_id": subj_id,
        })

print(f"Trial totali : {len(records)}")
print(f"Canali       : {len(keep_idx)}")


Trial totali : 38883
Canali       : 59


In [3]:
# ── Funzioni costruzione grafo (identiche a EEG_08) ─────────

def knn_from_matrix(matrix, k):
    edges = set()
    for i in range(matrix.shape[0]):
        for j in np.argsort(matrix[i])[::-1][:k]:
            edges.add((i, int(j)))
            edges.add((int(j), i))
    src, dst = zip(*sorted(edges))
    return torch.tensor([list(src), list(dst)], dtype=torch.long)


def pcc_to_edge_index(x_np, k=6):
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return knn_from_matrix(pcc, k)


def plv_to_edge_index(x_np, k=6, sfreq=256, bands=None):
    if bands is None:
        bands = PLV_BANDS
    nyq = sfreq / 2.0
    plv_sum = np.zeros((x_np.shape[0], x_np.shape[0]), dtype=np.float64)
    for flo, fhi in bands:
        b, a    = butter(4, [flo / nyq, fhi / nyq], btype="band")
        x_filt  = filtfilt(b, a, x_np, axis=1).astype(np.float32)
        analytic = sp_hilbert(x_filt, axis=1)
        exp_phi  = np.exp(1j * np.angle(analytic))
        plv_sum += np.abs(exp_phi @ exp_phi.conj().T) / x_np.shape[1]
    plv_matrix = (plv_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(plv_matrix, 0.0)
    return knn_from_matrix(plv_matrix, k)


def wpli_to_edge_index(x_np, k=6, sfreq=256, bands=None):
    if bands is None:
        bands = PLV_BANDS
    nyq = sfreq / 2.0
    wpli_sum = np.zeros((x_np.shape[0], x_np.shape[0]), dtype=np.float64)
    for flo, fhi in bands:
        b, a     = butter(4, [flo / nyq, fhi / nyq], btype="band")
        x_filt   = filtfilt(b, a, x_np, axis=1).astype(np.float32)
        analytic = sp_hilbert(x_filt, axis=1)
        imag_cs  = np.imag(
            analytic[:, :, np.newaxis] * analytic[np.newaxis, :, :].conj()
        )
        wpli_sum += np.abs(imag_cs.mean(axis=-1)) / (np.abs(imag_cs).mean(axis=-1) + 1e-8)
    wpli_matrix = (wpli_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(wpli_matrix, 0.0)
    return knn_from_matrix(wpli_matrix, k)


def sentinel_edge_index(x_np, k=6):
    """
    Placeholder edge_index per metodi learned e dynamic.
    Il grafo vero viene costruito dentro forward() — qui serve solo x.
    Ritorna edge_index minimo valido (self-loops su tutti i nodi).
    """
    n = x_np.shape[0]
    idx = list(range(n))
    return torch.tensor([idx, idx], dtype=torch.long)


GRAPH_FN = {
    "pcc":     pcc_to_edge_index,
    "plv":     plv_to_edge_index,
    "wpli":    wpli_to_edge_index,
    "learned": sentinel_edge_index,
    "dynamic": sentinel_edge_index,
}
print("Funzioni grafo OK")

Funzioni grafo OK


In [ ]:
# ============================================================
# ⚠️  BUILD DEPRECATO — NON ESEGUIRE
#
# Questo notebook produce dataset_*.pt con x.shape=(59, 384)
# perché rimuove A1/A2 durante il caricamento CSV.
# La pipeline usa esclusivamente il formato (61, 384) da EEG_07e.
#
# Per rigenerare i grafi usa: notebooks/EEG_07e_build_graphs_tensors.ipynb
# ============================================================

raise RuntimeError(
    "EEG_07b è DEPRECATO.\n"
    "Produce dataset_*.pt con 59 canali (A1/A2 rimossi) — formato non più usato.\n"
    "Usa EEG_07e per costruire graph_*.pt e hgraph_*.pt con 61 canali."
)


In [5]:
# ── Verifica ─────────────────────────────────────────────────
print(f"File in {GRAPHS_DIR}:")
for method in METHODS:
    out_path = GRAPHS_DIR / f"dataset_{method}_k{K_GRAPH}.pt"
    if out_path.exists():
        dl = torch.load(out_path, weights_only=False)
        d0 = dl[0]
        size_gb = out_path.stat().st_size / 1e9
        print(f"  {out_path.name:<35} {len(dl):>6} grafi  {size_gb:.2f} GB")
        print(f"    x: {tuple(d0.x.shape)}  edge_index: {tuple(d0.edge_index.shape)}  y: {d0.y.item()}  subj: {d0.subj.item()}")
    else:
        print(f"  {out_path.name} — NON TROVATO")

File in /home/daniele_u/miralis-hypergraph-imagined-speech/data/interim/graphs:
  dataset_pcc_k6.pt                    38883 grafi  3.87 GB
    x: (59, 384)  edge_index: (2, 508)  y: 0  subj: 0
  dataset_plv_k6.pt                    38883 grafi  3.87 GB
    x: (59, 384)  edge_index: (2, 486)  y: 0  subj: 0
  dataset_wpli_k6.pt                   38883 grafi  3.87 GB
    x: (59, 384)  edge_index: (2, 454)  y: 0  subj: 0
  dataset_cpcc_abs_k6.pt               38883 grafi  3.88 GB
    x: (59, 384)  edge_index: (2, 534)  y: 0  subj: 0
  dataset_cpcc_im_k6.pt                38883 grafi  3.90 GB
    x: (59, 384)  edge_index: (2, 502)  y: 0  subj: 0
  dataset_learned_k6.pt                38883 grafi  3.87 GB
    x: (59, 384)  edge_index: (2, 454)  y: 0  subj: 0
  dataset_dynamic_k6.pt                38883 grafi  3.87 GB
    x: (59, 384)  edge_index: (2, 454)  y: 0  subj: 0
